In [1]:
!pip install -q openai-agents yfinance resend schedule


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 850.8/850.8 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 10.0 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")
os.environ["RESEND_API_KEY"] = getpass("Enter Resend API Key: ")

Enter OpenAI API Key: ··········
Enter Resend API Key: ··········


In [3]:
import yfinance as yf

def get_stock_info(ticker):

    stock = yf.Ticker(ticker)

    info = stock.history(period="2d")

    current = info["Close"].iloc[-1]

    previous = info["Close"].iloc[-2]

    change = (current - previous) / previous * 100

    return {
        "ticker": ticker,
        "current": current,
        "previous": previous,
        "change": change
    }


In [4]:
from agents import Agent

agent = Agent(

    name="Stock Monitor",

    instructions="""
    You summarize stock price movements professionally.
    Keep the summary under 100 words.
    """,

    model="gpt-5.5"

)

In [6]:
import resend

resend.api_key = os.environ["RESEND_API_KEY"]

def send_email(subject, body):

    resend.Emails.send({

        "from": "onboarding@resend.dev",

        "to": "youremail@gmail.com",

        "subject": subject,

        "html": body

    })

In [12]:
from agents import Runner

THRESHOLD = 1.0

async def check_stock():

    stock = get_stock_info("MU")

    print(stock)

    if abs(stock["change"]) >= THRESHOLD:

        prompt = f"""

        Micron current price: {stock['current']}

        Previous close: {stock['previous']}

        Percentage change: {stock['change']:.2f}%

        Summarize this movement.

        """

        result = await Runner.run(

            agent,

            prompt

        )

        send_email(

            f"MU Alert ({stock['change']:.2f}%)",

            result.final_output

        )

        print("Email Sent!")

    else:

        print("Threshold not reached.")

In [13]:
import nest_asyncio
import schedule
import time
import asyncio

nest_asyncio.apply()

schedule.every(1).minutes.do(
    lambda: asyncio.create_task(check_stock())
)

while True:
    schedule.run_pending()
    time.sleep(1)

{'ticker': 'MU', 'current': np.float64(1133.989990234375), 'previous': np.float64(1043.18994140625), 'change': np.float64(8.704076335870717)}
{'ticker': 'MU', 'current': np.float64(1133.989990234375), 'previous': np.float64(1043.18994140625), 'change': np.float64(8.704076335870717)}
{'ticker': 'MU', 'current': np.float64(1133.989990234375), 'previous': np.float64(1043.18994140625), 'change': np.float64(8.704076335870717)}
Email Sent!
Email Sent!
Email Sent!
{'ticker': 'MU', 'current': np.float64(1133.989990234375), 'previous': np.float64(1043.18994140625), 'change': np.float64(8.704076335870717)}
{'ticker': 'MU', 'current': np.float64(1133.989990234375), 'previous': np.float64(1043.18994140625), 'change': np.float64(8.704076335870717)}
{'ticker': 'MU', 'current': np.float64(1133.989990234375), 'previous': np.float64(1043.18994140625), 'change': np.float64(8.704076335870717)}
{'ticker': 'MU', 'current': np.float64(1133.989990234375), 'previous': np.float64(1043.18994140625), 'change': n

KeyboardInterrupt: 